# 广播机制深入理解：不同形状的数组也能一起算

> 这是「数据分析从入门到精通」系列的第 4 篇。上篇学会了怎么从数组里精准取数，这篇来解决一个更烧脑的问题——两个形状不一样的数组，凭什么能直接相加？

---

嗨，我是小荷。

又见面了。上次有人说我迷萧何这事挺有意思，确实，我对这位汉初第一后勤部长是发自内心的佩服。你想啊，楚汉争霸那会儿，刘邦在前线打仗，萧何在后方统筹——**不同战场、不同节奏，但数据得对齐**，粮草该送多少、兵马该调多少，全靠萧何算得准。

NumPy 的广播机制，本质上就是在干这件事：**让不同"规格"的数据能够协同运算**。听起来有点抽象？别急，我用人话给你讲明白。

---

## 先从一个"奇怪"的现象说起

看这段代码，你觉得会报错吗？

In [1]:
import numpy as np

a = np.array([1, 2, 3])      # 形状 (3,)
b = np.array([[1], [2]])     # 形状 (2, 1)

result = a + b
print(result)
# [[2 3 4]
#  [3 4 5]]


[[2 3 4]
 [3 4 5]]


一个是一维数组，一个是二维数组，居然能直接相加？而且结果还是个 2×3 的矩阵？

这就是广播（Broadcasting）在起作用——**NumPy 自动帮你把数据"拉伸"对齐了**。

---

## 广播的核心规则（记住这两句话就够了）

规则其实不复杂，就两条：

**1. 维度对齐：从后往前比**

两个数组的 shape 从最后一个维度开始比较，如果相等，或者其中一个为 1，就能广播。

In [ ]:
# 例子
A: (5, 4, 3)
B:    (4, 1)  → 补成 (1, 4, 1)

从后往前比较：
- 3 vs 1 ✓ (可以广播，1会被拉伸)
- 4 vs 4 ✓
- 5 vs 1 ✓ (可以广播)
结果 shape: (5, 4, 3)


**2. 长度为 1 的维度会被"拉伸"**

就像复印机一样，把长度为 1 的维度复制成需要的数量。

In [2]:
a = np.array([[1], [2], [3]])   # shape (3, 1)
b = np.array([10, 20, 30])      # shape (3,)

# 广播后相加
a + b
# [[11 21 31]
#  [12 22 32]
#  [13 23 33]]


array([[11, 21, 31],
       [12, 22, 32],
       [13, 23, 33]])

---

## 实战场景：批量标准化不同班级的成绩

假设你有三个班的成绩，要分别减去各自班级的平均分，广播机制让这事变得超简单：

In [4]:
import numpy as np

# 3个班，每班4个学生的成绩
scores = np.array([
    [85, 90, 78, 92],   # 一班
    [70, 75, 80, 72],   # 二班
    [88, 85, 90, 87]    # 三班
])

# 计算每个班的平均分，reshape成(3,1)用于广播
print(scores.mean(axis=1).shape)
class_mean = scores.mean(axis=1).reshape(-1, 1)

# 广播减法：每个成绩减去自己班的平均分
standardized = scores - class_mean
print(standardized.round(2))
# [[-1.25  3.75 -8.25  5.75]
#  [-4.25  0.75  5.75 -2.25]
#  [ 0.5  -2.5   2.5  -0.5 ]]


(3,)
[[-1.25  3.75 -8.25  5.75]
 [-4.25  0.75  5.75 -2.25]
 [ 0.5  -2.5   2.5  -0.5 ]]


不用写 for 循环，广播自动帮你搞定。**这就好比萧何统筹粮草，各路人马规格不同，但他总能调度得井井有条**。

---

## 常见的广播模式

| 场景 | 数组 A | 数组 B | 结果 | 用途 |
|------|--------|--------|------|------|
| 标量运算 | `(3, 4)` | `()` | `(3, 4)` | 全体加减乘除 |
| 行向量广播 | `(3, 4)` | `(4,)` | `(3, 4)` | 每列加减同一组数 |
| 列向量广播 | `(3, 4)` | `(3, 1)` | `(3, 4)` | 每行加减同一组数 |

In [5]:
# 行向量广播
matrix = np.array([[1, 2, 3], [4, 5, 6]])
weights = np.array([10, 100, 1000])
print(matrix * weights)
# [[  10  200 3000]
#  [  40  500 6000]]

[[  10  200 3000]
 [  40  500 6000]]


In [7]:
# 列向量广播
bias = np.array([[1], [2]])
print(matrix + bias)
# [[2 3 4]
#  [6 7 8]]

[[2 3 4]
 [6 7 8]]


---

## 什么时候会广播失败？

In [8]:
a = np.array([[1, 2, 3], [4, 5, 6]])  # (2, 3)
b = np.array([10, 20])                 # (2,)

a + b  # ❌ ValueError: 无法广播


ValueError: operands could not be broadcast together with shapes (2,3) (2,) 

**原因：**
```
A: (2, 3)
B: (1, 2)  ← 补1后

3 vs 2，不相等且都不是1 → 广播失败
```

**解决：**调整 shape，让维度能对上。

In [9]:
b = np.array([[10], [20]])  # (2, 1)
print(a + b)  # ✅ 成功


[[11 12 13]
 [24 25 26]]


---

## 实战：图片数据归一化

图像处理里经常要对 RGB 三通道分别做标准化：

In [10]:
# 模拟 100×100 的 RGB 图片
image = np.random.randint(0, 256, size=(100, 100, 3))

# 每个通道的均值 (3,)
mean_rgb = image.mean(axis=(0, 1))  # [R_mean, G_mean, B_mean]

# 广播减法：每个像素减去对应通道的均值
normalized = image - mean_rgb

print(f"原图范围: {image.min()} ~ {image.max()}")
print(f"归一化后: {normalized.min():.1f} ~ {normalized.max():.1f}")


原图范围: 0 ~ 255
归一化后: -127.9 ~ 128.1


---

## 本篇小结

| 概念 | 要点 |
|------|------|
| 广播是什么 | 让不同形状的数组能进行运算的机制 |
| 核心规则 | 从后往前对齐，维度相等或其中一个为1 |
| 常见用法 | 标量运算、行/列向量广播、批量标准化 |
| 注意陷阱 | 维度对不上时报错，用 reshape 调整 |

记住：**广播不是真的复制数据，而是虚拟拉伸，所以特别快**。

---

## 课后练习

In [12]:
import numpy as np

# 题目数据
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

# 任务1：A 的每一行都加上 [10, 20, 30]
# 任务2：A 的每一列都乘以 [1, 2, 3]（第1行×1，第2行×2...）
# 任务3：计算 A 每列的 z-score（减去列均值，除以列标准差）


In [14]:
# 任务1：A 的每一行都加上 [10, 20, 30]
B = [10,20,30]
A + B

array([[11, 22, 33],
       [14, 25, 36],
       [17, 28, 39]])

In [15]:
# 任务2：A 的每一列都乘以 [1, 2, 3]（第1行×1，第2行×2...）
C = [[1],[2],[3]]
A * C

array([[ 1,  2,  3],
       [ 8, 10, 12],
       [21, 24, 27]])

In [18]:
# 任务3：计算 A 每列的 z-score（减去列均值，除以列标准差）
A_mean = A.mean(axis=0)
A_std = A.std(axis=0)
(A - A_mean)/A_std

array([[-1.22474487, -1.22474487, -1.22474487],
       [ 0.        ,  0.        ,  0.        ],
       [ 1.22474487,  1.22474487,  1.22474487]])

把你的答案扔评论区，我看 👀

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 5 篇：NumPy 常用统计函数**
>
> 均值、方差、分位数、相关系数……这些统计指标用 NumPy 怎么算？下篇用**电商销售数据分析**的实战场景，把常用统计函数串讲一遍。

---

**觉得广播机制有点意思？**

👇 点「在看」，推给更多想学 NumPy 的人  
💬 评论区说说你遇到过哪些广播报错的坑  
⭐ 关注公众号，跟萧何迷妹一起学数据分析

---

*「数据分析从入门到精通」系列 · 第 4 篇*  
*上一篇：[第 3 篇：数组创建、索引与切片]*  
*下一篇：第 5 篇：NumPy 常用统计函数*